In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os, glob

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype']=42 # Ensure fonts are embedded as editable text
plt.rcParams['ps.fonttype']= 42 #Same for Eps files

In [ ]:
country = 'US'
city = 'SanFrancisco'

files1 = glob.glob(f'../../data/regression_outputs_new/Fold/{country}/{city}/Fuse/Multi_Concat/results.csv')
files2 = glob.glob(f'../../data/regression_outputs_new/Fold/{country}/{city}/Fuse/Multi_Concat/result.csv')
files = files1 + files2
len(files)

In [ ]:
meta = pd.read_csv(f'../../data/processed/0labels/{country}.csv')
meta

In [ ]:
targets = meta['ID'].unique().tolist()

In [ ]:
ls = []
for i, file in enumerate(files):
    model = file.split('/')[-2]
    tmp = pd.read_csv(file)
    tmp.set_index('target', inplace=True)
    targets = list(set(targets) & set(tmp.index))
    tmp = tmp.loc[targets]
    if 'avg_r2' in tmp.columns:
        tmp = tmp[['avg_r2']]
        tmp = tmp.rename(columns={'avg_r2': model})
    else:
        tmp = tmp[['r2']]
        tmp = tmp.rename(columns={'r2': model})
    ls.append(tmp)
df = pd.concat(ls, axis=1)
df.reset_index(inplace=True)
df = df.merge(meta, left_on='target', right_on='ID', how='left').drop(columns=['ID'])
df = df[df['Multi_Concat'] > 0]
df.sort_values(by='SDG', inplace=True)
df

In [ ]:
sdg_colors = {
    1: "#e5243b",
    3: "#4C9F38",
    4: "#C5192D",
    5: "#FF3A21",
    6: "#26BDE2",
    8: "#A21942",
    9: "#FD6925",
    10: "#DD1367",
    11: "#FD9D24",
    13: "#3F7E44",
    16: "#00689D"
}

fig, ax = plt.subplots(figsize=(18/2.5, 8/2.5))
sns.barplot(data=df, x='Code', y='Multi_Concat', hue='SDG', palette=sdg_colors, dodge=False, width=0.5, ax=ax)

# Keep only the left and bottom spines.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)  # keep left spine
ax.spines['bottom'].set_visible(True)  # keep bottom spine
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

ax.tick_params(axis='both', labelsize=7,color='black')
ax.set_ylabel(r"$R^2$", fontsize=8,color='black')
ax.set_xlabel("", fontsize=8,color='black')
ax.set_xticklabels(df['Code'], rotation=90, fontsize=7, color='black')
# ax.set_ylim(0, 0.6)
plt.legend(frameon=False, loc='upper right', fontsize=7)

# plt.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.1, 1), fontsize=7)
plt.savefig(f"../../data/figure_assets/S_{country}_{city}_fold.pdf", dpi=300, bbox_inches='tight')
plt.show()